# 04 — Download PBMC cytokines and inventory it (X not loaded)
`cellflow.datasets.pbmc_cytokines()` → `huggingface_hub.hf_hub_download(repo_id="theislab/cellflow-datasets",
filename="pbmc_parse.h5ad", repo_type="dataset")`. The HF cache defaults to `~/.cache/huggingface` (**$HOME**),
so `HF_HOME` is redirected to `/fast/AG_Bunina/Pavel_1/data/cellflow/hf_cache` **before** importing huggingface_hub.

In [1]:
import os
DATA = "/fast/AG_Bunina/Pavel_1/data/cellflow"
os.environ["HF_HOME"] = f"{DATA}/hf_cache"          # must be set before huggingface_hub is imported
os.environ["TMPDIR"] = f"{DATA}/tmp"
import huggingface_hub.constants as hfc
print("HF hub cache:", hfc.HF_HUB_CACHE)
assert hfc.HF_HUB_CACHE.startswith(DATA), "HF cache would land in $HOME!"

HF hub cache: /fast/AG_Bunina/Pavel_1/data/cellflow/hf_cache/hub


In [2]:
from huggingface_hub import HfApi
info = HfApi().get_paths_info("theislab/cellflow-datasets", ["pbmc_parse.h5ad"], repo_type="dataset")[0]
print(f"remote size: {info.size/1e9:.2f} GB  | sha256 (LFS): {info.lfs.sha256 if info.lfs else None}")

remote size: 13.10 GB  | sha256 (LFS): c72671b36248b82c559211a28fbf009f4f077e3b2097f9ba6a04d27d27edd4c9


In [3]:
import time, cellflow
from cellflow.datasets import pbmc_cytokines
t = time.time()
adata = pbmc_cytokines(backed="r")                   # downloads (once) and opens without loading X
print(f"loaded backed in {time.time()-t:.0f}s from", adata.filename)
print(f"on-disk size: {os.path.getsize(adata.filename)/1e9:.2f} GB ({os.path.getsize(adata.filename):,} bytes)")
!du -shL {adata.filename}; du -sh ~/.cache/huggingface 2>/dev/null || echo "nothing in ~/.cache/huggingface"

/fast/AG_Bunina/Pavel_1/miniforge3/envs/cellflow/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


loaded backed in 30s from /fast/AG_Bunina/Pavel_1/data/cellflow/hf_cache/hub/datasets--theislab--cellflow-datasets/snapshots/4b235c9a1e0073c95211c6ef6257db184fc83f71/pbmc_parse.h5ad
on-disk size: 13.10 GB (13,101,592,422 bytes)


13G	/fast/AG_Bunina/Pavel_1/data/cellflow/hf_cache/hub/datasets--theislab--cellflow-datasets/snapshots/4b235c9a1e0073c95211c6ef6257db184fc83f71/pbmc_parse.h5ad
nothing in ~/.cache/huggingface


In [4]:
print("n_obs =", f"{adata.n_obs:,}", "| n_vars =", f"{adata.n_vars:,}", "| isbacked =", adata.isbacked)
print("X type on disk:", type(adata.X))
print("\nobs columns:"); print(adata.obs.dtypes.to_string())
print("\nvar columns:", list(adata.var.columns)); print("layers:", list(adata.layers.keys()), "| obsm:", list(adata.obsm.keys()))

n_obs = 9,697,974 | n_vars = 2,000 | isbacked = True
X type on disk: <class 'anndata._core.sparse_dataset._CSRDataset'>

obs columns:
sample                     category
species                    category
gene_count                    int64
tscp_count                    int64
mread_count                   int64
bc1_wind                      int64
bc2_wind                      int64
bc3_wind                      int64
bc1_well                   category
bc2_well                   category
bc3_well                   category
log1p_n_genes_by_counts     float64
log1p_total_counts          float64
total_counts_MT             float64
pct_counts_MT               float64
log1p_total_counts_MT       float64
donor                      category
cytokine                   category
treatment                  category
cell_type                  category
cytokine_family            category

var columns: ['n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm']
layers: [] | obsm: []

In [5]:
import pandas as pd
pd.set_option("display.max_rows", 200)
for col in ["donor", "cytokine", "cell_type", "cytokine_family"]:
    if col not in adata.obs:
        print(f"!! column '{col}' not in obs"); continue
    u = sorted(map(str, adata.obs[col].unique()))
    print(f"\n### {col}: {len(u)} unique\n{u}")


### donor: 12 unique
['Donor1', 'Donor10', 'Donor11', 'Donor12', 'Donor2', 'Donor3', 'Donor4', 'Donor5', 'Donor6', 'Donor7', 'Donor8', 'Donor9']

### cytokine: 91 unique
['4-1BBL', 'ADSF', 'APRIL', 'BAFF', 'C3a', 'C5a', 'CD27L', 'CD30L', 'CD40L', 'CT-1', 'Decorin', 'EGF', 'EPO', 'FGF-beta', 'FLT3L', 'FasL', 'G-CSF', 'GDNF', 'GITRL', 'GM-CSF', 'HGF', 'IFN-alpha1', 'IFN-beta', 'IFN-epsilon', 'IFN-gamma', 'IFN-lambda1', 'IFN-lambda2', 'IFN-lambda3', 'IFN-omega', 'IGF-1', 'IL-1-alpha', 'IL-1-beta', 'IL-10', 'IL-11', 'IL-12', 'IL-13', 'IL-15', 'IL-16', 'IL-17A', 'IL-17B', 'IL-17C', 'IL-17D', 'IL-17E', 'IL-17F', 'IL-18', 'IL-19', 'IL-1Ra', 'IL-2', 'IL-20', 'IL-21', 'IL-22', 'IL-23', 'IL-24', 'IL-26', 'IL-27', 'IL-3', 'IL-31', 'IL-32-beta', 'IL-33', 'IL-34', 'IL-35', 'IL-36-alpha', 'IL-36Ra', 'IL-4', 'IL-5', 'IL-6', 'IL-7', 'IL-8', 'IL-9', 'LIF', 'LIGHT', 'LT-alpha1-beta2', 'LT-alpha2-beta1', 'Leptin', 'M-CSF', 'Noggin', 'OSM', 'OX40L', 'PBS', 'PRL', 'PSPN', 'RANKL', 'SCF', 'TGF-beta1', 'TL1


### cell_type: 18 unique
['B Intermediate/Memory', 'B Naive', 'CD14 Mono', 'CD16 Mono', 'CD4 Memory', 'CD4 Naive', 'CD8 Memory', 'CD8 Naive', 'HSPC', 'ILC', 'MAIT', 'NK', 'NK CD56bright', 'NKT', 'Plasmablast', 'Treg', 'cDC', 'pDC']

### cytokine_family: 15 unique
['Chemokine', 'Common β chain', 'Common γ chain/IL-13/TSLP', 'Complement', 'Growth factor', 'IL-1', 'IL-10', 'IL-10 (IL-20 subfamily)', 'IL-17', 'IL-6/IL-12', 'Interferon', 'Interleukin - non-classified', 'Other', 'TNF', 'nan']


In [6]:
print("uns keys:", list(adata.uns.keys()))

uns keys: ['donor_embeddings', 'esm2_embeddings', 'hvg', 'log1p']


In [7]:
import numpy as np
emb = adata.uns["esm2_embeddings"]
print("type:", type(emb))
if isinstance(emb, dict):
    k0 = next(iter(emb)); v0 = np.asarray(emb[k0])
    print("n entries:", len(emb), "| vector dim:", v0.shape, v0.dtype, "| example key:", k0)
    print("keys:", sorted(emb.keys()))
    print("all dims equal:", len({np.asarray(v).shape for v in emb.values()}) == 1)
else:
    print("shape:", getattr(emb, "shape", None))

type: <class 'dict'>
n entries: 93 | vector dim: (2560,) float64 | example key: 4-1BBL
keys: ['4-1BBL', 'ADSF', 'APRIL', 'BAFF', 'C3a', 'C5a', 'CD27L', 'CD30L', 'CD40L', 'CT-1', 'Decorin', 'EBI3', 'EGF', 'EPO', 'FGF-beta', 'FLT3L', 'FasL', 'G-CSF', 'GDNF', 'GITRL', 'GM-CSF', 'HGF', 'IFN-alpha1', 'IFN-beta', 'IFN-epsilon', 'IFN-gamma', 'IFN-lambda1', 'IFN-lambda2', 'IFN-lambda3', 'IFN-omega', 'IGF-1', 'IL-1-alpha', 'IL-1-beta', 'IL-10', 'IL-11', 'IL-12', 'IL-13', 'IL-15', 'IL-16', 'IL-17A', 'IL-17B', 'IL-17C', 'IL-17D', 'IL-17E', 'IL-17F', 'IL-18', 'IL-19', 'IL-1Ra', 'IL-2', 'IL-20', 'IL-21', 'IL-22', 'IL-23', 'IL-24', 'IL-26', 'IL-27', 'IL-3', 'IL-31', 'IL-32-beta', 'IL-33', 'IL-34', 'IL-35', 'IL-36-alpha', 'IL-36Ra', 'IL-4', 'IL-5', 'IL-6', 'IL-7', 'IL-8', 'IL-9', 'LIF', 'LIGHT', 'LT-alpha', 'LT-alpha1-beta2', 'LT-alpha2-beta1', 'LT-beta', 'Leptin', 'M-CSF', 'Noggin', 'OSM', 'OX40L', 'PRL', 'PSPN', 'RANKL', 'SCF', 'TGF-beta1', 'TL1A', 'TNF-alpha', 'TPO', 'TRAIL', 'TSLP', 'TWEAK', 'VEG

In [8]:
# other uns entries: types/sizes (for the record)
for k, v in adata.uns.items():
    desc = f"dict[{len(v)}]" if isinstance(v, dict) else getattr(v, "shape", type(v).__name__)
    print(f"{k:35s} {type(v).__name__:12s} {desc}")

donor_embeddings                    dict         dict[12]
esm2_embeddings                     dict         dict[93]
hvg                                 dict         dict[1]
log1p                               dict         dict[0]


In [9]:
# counts useful for choosing the subset
print(pd.crosstab(adata.obs["donor"], adata.obs["cytokine"] == "PBS").to_string())
print(adata.obs["cytokine"].value_counts().to_string())

cytokine    False  True 
donor                   
Donor1    1177238  90452
Donor2     861049  70520
Donor3     646591  48567
Donor4     533724  44518
Donor5    1011432  55293
Donor6     914872  59158
Donor7     614570  37816
Donor8     607910  43064
Donor9     603012  29329
Donor10    544936  35850
Donor11    569410  43955
Donor12    983529  71179
cytokine
PBS                629701
CT-1               136899
IL-36-alpha        135264
IL-9               130692
TSLP               129206
IL-7               126167
IL-36Ra            126129
IL-33              124191
IL-13              122162
G-CSF              117834
IL-23              116852
IL-21              114669
SCF                114595
OSM                113323
IFN-alpha1         113165
IL-15              113033
LIGHT              112956
CD27L              111598
IL-4               110998
IL-34              109886
IL-17F             109831
4-1BBL             109213
IFN-epsilon        109078
ADSF               108142
Noggin           